# 01 — Data Exploration
Load both datasets, inspect, build corpus files at 1k / 5k / 20k docs.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
os.chdir('..')

In [ ]:
from data.loader import load_dataset, build_corpus
from config import SMOKE_CORPUS_SIZE, CORPUS_SIZES, CORPUS_DIR
import json, os

## Load FinQA

In [ ]:
finqa_train = load_dataset('finqa', split='train', max_samples=500)
finqa_test  = load_dataset('finqa', split='test',  max_samples=200)
print(f'FinQA train: {len(finqa_train)} | test: {len(finqa_test)}')
print('Sample:', finqa_train[0])

## Load MultiHop-RAG

In [ ]:
multihop_train = load_dataset('multihop', split='train', max_samples=500)
print(f'MultiHop train: {len(multihop_train)}')
print('Sample:', multihop_train[0])

## Build Corpus at Multiple Sizes

In [ ]:
# For large corpus sizes we need more samples
finqa_large    = load_dataset('finqa', split='train')
multihop_large = load_dataset('multihop', split='train')
all_samples    = finqa_large + multihop_large
print(f'Total samples: {len(all_samples)}')

In [ ]:
os.makedirs(CORPUS_DIR, exist_ok=True)
for size in [1000, 5000, 20000]:
    corpus = build_corpus(all_samples, corpus_size=size)
    path   = os.path.join(CORPUS_DIR, f'corpus_{size}.json')
    with open(path, 'w') as f:
        json.dump(corpus, f)
    print(f'Saved corpus_{size}.json — {len(corpus)} docs')

## Chunking Stats

In [ ]:
from pipeline.chunker import chunk_documents
import json

with open(os.path.join(CORPUS_DIR, 'corpus_1000.json')) as f:
    corpus_1k = json.load(f)

chunks = chunk_documents(corpus_1k)
print(f'1k docs -> {len(chunks)} chunks')
print(f'Sample chunk: {chunks[0]}')

## Embedding Shape Verification

In [ ]:
from pipeline.embedder import embed_chunks
sample_chunks = chunks[:50]
emb = embed_chunks(sample_chunks)
print(f'Embeddings shape: {emb.shape}')  # expect (50, 384)

## Save Eval Question Samples

In [ ]:
from config import EVAL_SAMPLE_SIZE, ACCURACY_DIR
os.makedirs(ACCURACY_DIR, exist_ok=True)

finqa_eval = finqa_test[:EVAL_SAMPLE_SIZE]
with open(os.path.join(ACCURACY_DIR, 'finqa_eval_questions.json'), 'w') as f:
    json.dump(finqa_eval, f)

multihop_eval = multihop_train[:EVAL_SAMPLE_SIZE]
with open(os.path.join(ACCURACY_DIR, 'multihop_eval_questions.json'), 'w') as f:
    json.dump(multihop_eval, f)

print(f'Saved {len(finqa_eval)} FinQA eval + {len(multihop_eval)} MultiHop eval questions')